# Performance Tuning

## What's covered

- Partition sizing — the unit of parallelism, and how to right-size it
- `repartition` vs `coalesce` vs `partitionBy` — three different things that share a name
- Tuning `spark.sql.shuffle.partitions`
- What triggers a shuffle, and reading shuffles in `.explain()`
- Catalyst's automatic wins: predicate pushdown + column pruning
- Adaptive Query Execution (AQE) — runtime re-optimization, including skew join
- Caching DataFrames: `cache()`, `persist()`, storage levels (and why `cache()` is lazy)
- `cacheTable`, `unpersist`, and when NOT to cache
- Checkpointing — truncating long lineage
- Driver-side bottlenecks: `collect()` traps and the `withColumn`-in-a-loop catastrophe
- Join strategies: broadcast hash, sort-merge, shuffle hash — and when NOT to broadcast
- Detecting and fixing data skew: AQE skew join, salting, `repartitionByRange`


## Partition sizing — the unit of parallelism

A partition is processed by exactly one task on one executor core. Two failure modes:

- **Too few partitions** — cores sit idle; one slow task stalls the stage; risk of OOM if a partition is huge.
- **Too many partitions** — task scheduling overhead dominates; tiny tasks finish in milliseconds while the scheduler spends longer setting them up than the work itself.

Rules of thumb:

- **Production / cluster** — target **128–256 MB per partition**.
- **Local mode** — **2–4× the number of CPU cores** is plenty.
- **After a shuffle** — the count is controlled by `spark.sql.shuffle.partitions` (default 200). Tune in the section below.

You can ask a DataFrame how many partitions it currently has via `df.rdd.getNumPartitions()` and inspect the row distribution with `spark_partition_id()`.


In [ ]:
import os
import time
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, spark_partition_id, count

spark = (
    SparkSession.builder
    .appName("PerformanceTuning")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "data"
card_txns  = spark.read.parquet(f"{DATA_DIR}/card_transactions")
customers  = spark.read.option("header", "true").csv(f"{DATA_DIR}/customers")
loan_accts = spark.read.json(f"{DATA_DIR}/loan_accounts")

print("card_transactions :", card_txns.count(), "rows,",
      card_txns.rdd.getNumPartitions(), "partitions")
print("customers         :", customers.count(), "rows")
print("loan_accounts     :", loan_accts.count(), "rows")

# Inspect the row distribution across partitions
(
    card_txns
    .withColumn("pid", spark_partition_id())
    .groupBy("pid").agg(count("*").alias("n"))
    .orderBy("pid")
    .show()
)


## `repartition` vs `coalesce` vs `partitionBy` — three different things

Three operations all have "partition" in the name and are routinely confused. They do **completely different jobs**:

| | What it does | Where it lives | Shuffle |
|---|---|---|---|
| `df.repartition(n)` / `df.repartition(n, col)` | Redistribute rows across `n` partitions in memory (optionally hash-partition by column) | Anywhere in the DataFrame pipeline | **Full shuffle** |
| `df.coalesce(n)` | Merge existing partitions down to `n` without redistributing | Anywhere — but only to **decrease** count | **No shuffle** (narrow) |
| `df.write.partitionBy("col")` | At **write time**, create one **subdirectory** per distinct value of `col` (Hive-style on-disk layout) | Only on `.write` | No in-memory effect; changes file layout only |

The runtime-vs-write-time split is the key. `repartition` and `coalesce` change how rows are laid out **in memory** for the next stage. `partitionBy` changes how rows are laid out **on disk** for downstream readers (it's covered in nb 04).

The third form — **`repartition(n, col)`** — hash-partitions in memory by a column. All rows with the same key value land in the same partition. Use it before a join on that column to **eliminate the shuffle at join time**.


In [ ]:
# Three reshapes of the same DataFrame
print("original         :", card_txns.rdd.getNumPartitions())
print("repartition(16)  :", card_txns.repartition(16).rdd.getNumPartitions())
print("coalesce(2)      :", card_txns.coalesce(2).rdd.getNumPartitions())

# repartition(n, col) — hash-partition by a column for an upcoming join
keyed = card_txns.repartition(8, "customer_id")
(
    keyed
    .withColumn("pid", spark_partition_id())
    .groupBy("pid").agg(count("*").alias("n"))
    .orderBy("pid")
    .show()
)


## Tuning `spark.sql.shuffle.partitions`

The default — **200** — is sized for large clusters. On `local[*]` with 8 cores that's 192 mostly-empty partitions and a lot of scheduling overhead. AQE coalesces empty post-shuffle partitions automatically (notebook 01), but on large datasets you may still want to set the upstream count explicitly.

**Rule of thumb**: 2–4× total core count for local; for a real cluster, target the 128–256 MB-per-partition rule on the largest expected post-shuffle dataset.


In [ ]:
def time_groupby(n):
    spark.conf.set("spark.sql.shuffle.partitions", n)
    t0 = time.perf_counter()
    out = (
        card_txns
        .filter(col("status") == "APPROVED")
        .groupBy("merchant_category")
        .count()
    )
    out.count()  # action
    return time.perf_counter() - t0, out.rdd.getNumPartitions()

t1, p1 = time_groupby(200)
t2, p2 = time_groupby(8)
print(f"shuffle.partitions=200 -> {t1:.2f}s, {p1} output partitions")
print(f"shuffle.partitions=8   -> {t2:.2f}s, {p2} output partitions")

# Restore default for downstream cells
spark.conf.set("spark.sql.shuffle.partitions", 200)


## What triggers a shuffle

A shuffle happens when Spark must redistribute data across partitions. Three steps every time:

1. **Shuffle write** — each task writes intermediate output to local disk.
2. **Network transfer** — data moves to target executors.
3. **Shuffle read** — receiving tasks read and deserialize.

Operations that **always** shuffle: `groupBy`, `join`, `distinct`, `orderBy`, `repartition`, `dropDuplicates`, `union(distinct=True)`.

Operations that **never** shuffle (narrow): `filter`, `select`, `withColumn`, `coalesce` (when reducing), `map`, `flatMap`.

![Shuffle Anatomy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-shuffle-anatomy.svg)


## Reading shuffles in `explain()`

Each shuffle shows up as an `Exchange` operator in the physical plan. Counting `Exchange` nodes is the fastest way to estimate the network cost of a query.


In [ ]:
# Single-table groupBy → one Exchange
single = card_txns.filter(col("status") == "APPROVED").groupBy("merchant_category").count()
print("=== single-table groupBy ===")
single.explain()

# Join + groupBy → two Exchange operators (one per side of the join) plus one for the groupBy
joined = card_txns.join(customers, "customer_id").groupBy("city").count()
print("\n=== join + groupBy ===")
joined.explain()


## Catalyst's free wins — predicate pushdown + column pruning

For columnar formats (Parquet, ORC), Catalyst applies two optimizations *automatically* — no code change required.

- **Predicate pushdown** — filters travel into the file reader. The reader skips entire row groups whose statistics cannot match the predicate, without ever decoding them.
- **Column pruning** — Catalyst tracks which columns are actually referenced and tells the file reader to skip the rest. Unread columns are never decoded.

Together, these turn a "read 10 GB and filter" query into a "read 100 MB" query — without any user-side change.

You can confirm both via `.explain(mode="extended")` and look at:

- **`PushedFilters`** — the predicates the reader pushed into Parquet/ORC.
- **`ReadSchema`** — only the columns Catalyst actually needs.

*Full breakdown of what does/doesn't push lives in nb 04 (Reading & Writing).*


In [ ]:
# A query with both pushdown and pruning opportunities
optimized = (
    card_txns
    .filter(col("status") == "APPROVED")
    .filter(col("amount") > 10000)
    .select("transaction_id", "amount")
)

# Look for:
#   PushedFilters: [..., GreaterThan(amount,10000)]
#   ReadSchema: only the columns we projected, not the full schema
optimized.explain(mode="extended")


## AQE in depth — runtime re-optimization

Notebook 01 introduced AQE with **dynamic coalescing of post-shuffle partitions**. Two more behaviors round out the set:

- **Dynamic join strategy switch** — at runtime, after a shuffle's stats are known, Spark can promote a sort-merge join to a broadcast hash join if one side turns out small enough. Threshold: `spark.sql.adaptive.autoBroadcastJoinThreshold`.
- **Skew join handling** — Spark detects oversized partitions and *splits* them into sub-tasks so one slow task doesn't stall the stage. A partition is considered skewed when:
    - Its size is ≥ `spark.sql.adaptive.skewJoin.skewedPartitionFactor` × the median (default 5×), **and**
    - Its size is ≥ `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default 256 MB).

AQE is on by default in Spark 3.2+. We use skew handling end-to-end in the skew section below.


In [ ]:
print("aqe enabled              :", spark.conf.get("spark.sql.adaptive.enabled"))
print("skew join enabled        :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("skew factor              :", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor"))
print("skew threshold (bytes)   :", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes"))


## Caching DataFrames — `cache()` and `persist()`

**Spark's default behavior — every action re-runs the entire lineage from the source.** This is the first performance trap to internalize. If you call `df.count()` and then `df.collect()` on the same DataFrame, the read-and-transform pipeline runs *twice*. Spark doesn't remember the result; only the recipe.

Caching fixes this by materializing once and reusing.

- **`df.cache()`** — shorthand for `df.persist(StorageLevel.MEMORY_AND_DISK)`. **Lazy** — calling `cache()` alone does **nothing**. The next action materializes the result; subsequent actions read the cached copy. So `df.cache(); df.show()` is *two* operations: the first marks the DataFrame as cacheable, the second triggers materialization.
- **`df.persist(level)`** — pick the storage policy explicitly.

**Key note — `cache()` is lazy.** Two consequences worth remembering:

- A `cache()` call you "forgot to use" costs nothing — Spark only materializes on the next action.
- If you call `cache()` and never trigger an action, the cache is never populated, no matter how often you check `df.is_cached`.

> **Note** — DataFrame `cache()` defaults to `MEMORY_AND_DISK`. RDD `cache()` (notebook 02) defaults to `MEMORY_ONLY`. The DataFrame default safely spills to disk if it doesn't fit in RAM; the RDD default just recomputes on miss.


In [ ]:
def expensive():
    return (
        spark.read.parquet(f"{DATA_DIR}/card_transactions")
        .filter(col("status") == "APPROVED")
        .filter(col("amount") > 1000)
        .select("customer_id", "merchant_category", "amount")
    )

# Without cache — every action re-runs the whole pipeline.
no_cache = expensive()
t0 = time.perf_counter()
no_cache.count()
no_cache.count()
print(f"no cache : two .count() calls in {time.perf_counter() - t0:.2f}s")

# With cache — first action materializes; second hits memory.
cached = expensive().cache()
t0 = time.perf_counter()
cached.count()  # materializes
cached.count()  # reads cache
print(f"cache    : two .count() calls in {time.perf_counter() - t0:.2f}s")

print("is_cached     :", cached.is_cached)
print("storage level :", cached.storageLevel)

cached.unpersist()


## Storage levels — reference

`persist()` accepts a `StorageLevel` to control where the data lives.

| StorageLevel | RAM | Disk | Serialized | Replicas | Best for |
|---|:-:|:-:|:-:|:-:|---|
| `MEMORY_ONLY` | ✓ | — | No | 1 | Small DataFrames that fit in RAM (RDD default) |
| `MEMORY_AND_DISK` | ✓ | ✓ (spill) | No | 1 | **Default for DataFrames** — safe for any size |
| `MEMORY_ONLY_SER` | ✓ | — | Yes | 1 | Large DataFrames; saves RAM at the cost of CPU |
| `MEMORY_AND_DISK_SER` | ✓ | ✓ | Yes | 1 | Large DataFrames with disk safety |
| `DISK_ONLY` | — | ✓ | Yes | 1 | Low-memory environments |
| `MEMORY_AND_DISK_2` | ✓ | ✓ | No | 2 | Fast partition recovery without recompute |
| `OFF_HEAP` | (off-heap) | — | Yes | 1 | Avoid GC pressure (advanced) |

**Rule of thumb**: start with `MEMORY_AND_DISK`. Switch to `MEMORY_ONLY` when you know the data fits and want the fastest reads. Use `_2` only when recomputing from lineage is genuinely expensive (multi-hour aggregations).


In [ ]:
from pyspark import StorageLevel

# Persist with an explicit storage level
loan_accts.persist(StorageLevel.MEMORY_AND_DISK_SER)
loan_accts.count()  # materialize

print("loan_accts is_cached     :", loan_accts.is_cached)
print("loan_accts storage level :", loan_accts.storageLevel)

loan_accts.unpersist()


## `cacheTable`, `unpersist`, and anti-patterns

**Catalog-level caching** — `spark.catalog.cacheTable("name")` caches a registered view at the catalog level. Any query against that view name reads from the cached copy.

**`unpersist()`** — always call it when you're done. Cached data stays in memory until the session ends, LRU evicts it, or you release it explicitly. Don't rely on LRU.

**Anti-patterns — when NOT to cache:**

| Scenario | Why it hurts |
|---|---|
| DataFrame used in only one action | Memory consumed but never reused |
| Very large DataFrame that exceeds RAM | Constant eviction/recompute thrashing |
| DataFrame whose source changes between actions | Cached data becomes stale |
| Simple scan with no transformations | Source I/O is already cheap |
| Intermediate step in a long chain that is only used once | Cache the *final* result instead |

The right rule: **cache the result of expensive transformations, not raw reads.**


In [ ]:
# Catalog-level caching — visible to SQL queries by name
customers.createOrReplaceTempView("customers")
spark.catalog.cacheTable("customers")
print("customers cached?", spark.catalog.isCached("customers"))

spark.sql("SELECT city, COUNT(*) AS n FROM customers GROUP BY city ORDER BY n DESC LIMIT 5").show()

spark.catalog.uncacheTable("customers")
print("customers cached after uncache?", spark.catalog.isCached("customers"))


## Checkpointing — truncating long lineage

Caching keeps the lineage. Spark can still recompute from the source if a cached partition is lost. **Checkpointing** writes the DataFrame to reliable storage and *breaks the lineage* — Spark forgets how the data was produced and just reads the saved files.

When to checkpoint:

- **Iterative algorithms** (ML training loops with 100+ iterations) where the plan itself becomes a memory and performance problem
- **Streaming jobs** with stateful operations that accumulate lineage over time
- **Any DAG longer than ~20 shuffle stages**

Unlike `cache()`, **checkpointing is eager** — it executes immediately when called.


In [ ]:
# Set a checkpoint directory (must be set before .checkpoint())
checkpoint_dir = os.path.abspath(os.path.join(DATA_DIR, "_checkpoints"))
shutil.rmtree(checkpoint_dir, ignore_errors=True)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

# Build a derived DataFrame with a non-trivial plan
derived = (
    card_txns
    .filter(col("status") == "APPROVED")
    .join(customers, "customer_id")
    .groupBy("city")
    .count()
)

print("=== BEFORE CHECKPOINT — full lineage ===")
derived.explain()

# Checkpoint — eager; writes to disk and breaks the lineage
checkpointed = derived.checkpoint(eager=True)

print("\n=== AFTER CHECKPOINT — lineage truncated ===")
checkpointed.explain()


## Driver-side bottlenecks — `collect`, `withColumn` in a loop

A handful of patterns look fast but kill performance — not on the executors, but on the driver or the planner.

### `collect()` pulls every row to the driver

The DataFrame is distributed across executors. `df.collect()` undoes that — every row travels back to the driver process and lands in a single Python list. On large data this **OOMs the driver**. Always aggregate or `limit()` first.

```python
# DANGEROUS — pulls 100M rows to the driver
all_rows = big_df.collect()

# Safe — already aggregated to a tiny result
agg_rows = big_df.groupBy("category").count().collect()

# Safe — bounded sample
sample = big_df.limit(100).collect()
```

`toPandas()` has the same trap on the pandas side — covered in nb 03.

### `withColumn` in a loop is **exponentially** slow

This pattern is the most common surprise in this notebook:

```python
# BAD — produces a 100-layer-deep logical plan
for c in range(100):
    df = df.withColumn(f"c_{c}", col("amount") * c)
```

Every `withColumn` call returns a *new* DataFrame whose plan tree contains the entire prior plan. After 100 iterations you have a 100-deep nested `Project` operator. Catalyst analyzes the plan **each time** you add another layer — and analysis cost is super-linear in plan depth. A 100-column loop can take minutes; the equivalent single `select` runs in milliseconds.

The fix: build the columns once, apply them in a single `select`:

```python
# GOOD — one Project node, one analysis pass
new_cols = [col("amount").alias("c_0")] + [
    (col("amount") * c).alias(f"c_{c}") for c in range(1, 100)
]
df = df.select("*", *new_cols)
```

Same rule applies to `withColumnRenamed` in a loop, and any other "iterate-and-extend" pattern.


In [ ]:
# Demo: withColumn-in-a-loop vs single select
N = 50
base = card_txns.select("transaction_id", "amount").limit(1000).cache()
base.count()  # materialize so the cache is warm before we time

# BAD — loop of withColumn calls
t0 = time.perf_counter()
df_loop = base
for i in range(N):
    df_loop = df_loop.withColumn(f"c_{i}", col("amount") * i)
df_loop.count()
print(f"withColumn loop ({N} cols) : {time.perf_counter() - t0:.2f}s")

# GOOD — single select with all columns at once
t0 = time.perf_counter()
new_cols = [(col("amount") * i).alias(f"c_{i}") for i in range(N)]
df_select = base.select("*", *new_cols)
df_select.count()
print(f"single select ({N} cols)   : {time.perf_counter() - t0:.2f}s")

base.unpersist()


## Join strategies, broadcast, and when NOT to broadcast

Spark picks a join algorithm based on table sizes and hints.

| Strategy | When | Shuffle | Best for |
|---|---|---|---|
| **Broadcast Hash Join** | One side ≤ `autoBroadcastJoinThreshold` (default 10 MB) or `broadcast()` hint | **None** | Small dimension tables |
| **Sort-Merge Join** | Both sides large, no broadcast possible | Both sides | Large fact-to-fact joins |
| **Shuffle Hash Join** | One side fits in memory per partition | One side | Medium tables |
| **Cartesian / Nested Loop** | Cross join or no join key | Full cross product | Explicit cross-joins only |

The **broadcast hash join** is the single biggest win for star-schema queries — joining a large fact table with small dimension tables. The small side is shipped to every executor; the join runs locally with no shuffle.

**Forcing a broadcast** — wrap the small side with `broadcast()`. In SQL, the `/*+ BROADCAST(alias) */` hint does the same.

**When NOT to broadcast:**

| Table size | Recommendation |
|---|---|
| < 10 MB | Auto-broadcast (Spark default) |
| 10 MB – 100 MB | Force broadcast — safe on most clusters |
| 100 MB – 500 MB | Broadcast with caution; test on your cluster |
| > 500 MB | **Do not broadcast** — sort-merge or AQE |

Broadcasting a table that doesn't fit in executor memory causes OOM.

![Broadcast vs Sort-Merge](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-broadcast-join.svg)


In [ ]:
from pyspark.sql.functions import broadcast

print("autoBroadcastJoinThreshold:",
      spark.conf.get("spark.sql.autoBroadcastJoinThreshold"), "bytes")

# Force a broadcast — small dimension (customers) joined with large fact (card_txns)
forced = card_txns.join(broadcast(customers), "customer_id").groupBy("city").count()
print("\n=== forced broadcast — look for BroadcastHashJoin ===")
forced.explain()

# SQL form: /*+ BROADCAST(c) */
card_txns.createOrReplaceTempView("card_transactions")
customers.createOrReplaceTempView("customers")
sql_forced = spark.sql("""
    SELECT /*+ BROADCAST(c) */ c.city, COUNT(*) AS n
    FROM   card_transactions t JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY c.city
""")
print("\n=== SQL hint — same plan ===")
sql_forced.explain()

# Disable auto-broadcast — plan flips to SortMergeJoin with two Exchanges
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
no_broadcast = card_txns.join(customers, "customer_id").groupBy("city").count()
print("\n=== auto-broadcast disabled — SortMergeJoin + two Exchanges ===")
no_broadcast.explain()

# Restore default
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)


## Detecting and fixing data skew

**Data skew** occurs when a few keys dominate the partition distribution. Because a stage cannot complete until the last task finishes, one oversized partition stalls the entire stage.

**Symptoms:**

- One task runs 10–100× longer than the median (visible in **Spark UI → Stages**)
- One executor shows GC pressure or disk spill while others sit idle
- The stage finishes quickly except for the last one or two tasks

**Three fixes — pick one based on Spark version and severity:**

1. **AQE skew join** — modern default. Spark 3.2+ ships with `spark.sql.adaptive.skewJoin.enabled = true`. AQE detects skewed partitions at runtime and splits them into sub-tasks. **Try this first.**
2. **Salting** — manually break hot keys. Append a random suffix to the join key on the large side; explode the small side N times to match. The most reliable fix when AQE isn't enough.
3. **`repartitionByRange(n, col)`** — sort by the column and split into equal-sized ranges; partitions are balanced regardless of key frequency.

Skew also appears in **`groupBy`** — same disease, same medicine. Salt the key, aggregate twice (partial then final).


In [ ]:
# Build a synthetically skewed DataFrame: 80% of rows go to one hot key.
from pyspark.sql.functions import when, lit, concat, rand

skewed = (
    spark.range(0, 200_000)
    .withColumn(
        "customer_id",
        when(col("id") % 5 != 0, lit("CUST_HOT"))                                 # 80% hot
        .otherwise(concat(lit("CUST_"), (col("id") % 100).cast("string")))         # 20% spread
    )
    .withColumn("amount", (rand(seed=42) * 10000).cast("decimal(18,2)"))
)

# Per-key distribution — one key dominates
print("=== per-key counts (top 5) ===")
skewed.groupBy("customer_id").count().orderBy(col("count").desc()).show(5)

# Per-partition distribution after a shuffle by customer_id — one big partition
print("=== per-partition counts after shuffle ===")
(
    skewed.repartition(8, "customer_id")
    .withColumn("pid", spark_partition_id())
    .groupBy("pid").agg(count("*").alias("n"))
    .orderBy("pid")
    .show()
)


In [ ]:
# Fix 1 — AQE skew join (the modern default, no code change needed)
print("AQE enabled              :", spark.conf.get("spark.sql.adaptive.enabled"))
print("AQE skew join enabled    :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

# A small dimension covering CUST_0..CUST_99 plus the hot key
small_dim = spark.range(0, 100).select(
    concat(lit("CUST_"), col("id").cast("string")).alias("customer_id"),
    (col("id") * 100 + 5000).alias("credit_limit"),
).union(
    spark.createDataFrame([("CUST_HOT", 99999)], ["customer_id", "credit_limit"])
)

# Run the join — AQE detects the skewed partition at runtime and splits it.
# On a real cluster, Spark UI -> Stages shows the split tasks for the hot key.
joined = skewed.join(small_dim, "customer_id")
print("joined rows:", joined.count())


In [ ]:
# Fix 2 — Salting. Break the hot key into N artificial sub-keys.
from pyspark.sql.functions import explode, array

N_SALTS = 10

# Large side: append a random salt to make a composite join key
large_salted = (
    skewed
    .withColumn("salt", (rand(seed=7) * N_SALTS).cast("int"))
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt").cast("string")))
)

# Small side: explode each row into N_SALTS rows, one per salt value
small_salted = (
    small_dim
    .withColumn("salt", explode(array(*[lit(i) for i in range(N_SALTS)])))
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt").cast("string")))
)

salted_join = (
    large_salted
    .join(small_salted, "salted_key")
    .select(large_salted["customer_id"], "amount", "credit_limit")
)

print("salted-join row count:", salted_join.count())


In [ ]:
# Fix 3 — repartitionByRange. Split by value range; hot keys are spread across partitions.
range_partitioned = skewed.repartitionByRange(8, "customer_id")

(
    range_partitioned
    .withColumn("pid", spark_partition_id())
    .groupBy("pid").agg(count("*").alias("n"))
    .orderBy("pid")
    .show()
)


In [ ]:
# Reset configs and stop the session
spark.conf.set("spark.sql.shuffle.partitions", 200)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

shutil.rmtree(checkpoint_dir, ignore_errors=True)
spark.stop()
